# Critical Review of a Credit-Card Fraud Detection Tutorial

**Course:** Data Science in Cyber — Final Project

**Selected source:** *Enhancing Credit Card Fraud Detection Models using Machine
Learning and PCA* (public GitHub project,
`BalaElangovan/Enhancing-Credit-Card-Fraud-Detection-Models-using-Machine-Learning-and-PCA`).
The project balances the data with **SMOTE**, reduces dimensionality with **PCA**,
and compares Logistic Regression, Random Forest, and a Neural Network. It reports
**AUC-ROC ≈ 1.00** for Random Forest and the Neural Network and 0.9923 for Logistic
Regression.

**Dataset:** Kaggle / ULB *Credit Card Fraud Detection* (Dal Pozzolo et al., 2015):
284,807 European card transactions, 492 fraudulent (0.172%). Features `Time`,
`V1`–`V28` (PCA-anonymised), `Amount`, and the binary target `Class`.

**Central question of this notebook.** Is the reported near-perfect performance a
genuine property of the models, or an artefact of methodology? Our thesis is the
latter: resampling applied **before** the train/test split leaks information, and
accuracy/ROC-AUC are the wrong lenses for a 0.172%-prevalence problem. We
(1) reproduce the flawed pipeline to show the inflated scores, then (2) build a
leakage-free pipeline and report honest metrics (PR-AUC, MCC, F2) and the
false-positive / false-negative trade-off that matters operationally.

All randomness uses a fixed seed (`RANDOM_STATE = 42`). Heavy logic lives in the
documented helper modules under `src/`, keeping this notebook a readable narrative.

In [1]:
import sys
from pathlib import Path

# Make the project root importable whether the notebook is run from the repo root
# or from a subfolder.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

from src import data, eda, evaluate, features, models
from src.config import RANDOM_STATE, TARGET, PCA_FEATURES

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(RANDOM_STATE)
print("Environment ready. RANDOM_STATE =", RANDOM_STATE)

Environment ready. RANDOM_STATE = 42


## 1. Data Loading and Inspection

We first download the dataset on demand (it is too large to store in Git), then
load it and inspect its structure, types, and integrity.

In [2]:
data.download_data()  # no-op if data/creditcard.csv already exists

raw = data.load_data(drop_duplicates=False)
print("Raw shape:", raw.shape)
raw.head()

Raw shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


### Feature types, index, and column names

The index is a default `RangeIndex` (no meaningful key such as a transaction id),
and every column is numeric. The column names are uninformative by design: the
publisher applied PCA and released only the components `V1`–`V28` to protect
cardholder privacy. The only two human-readable features are `Time` (seconds since
the first transaction) and `Amount`. This is sensible for a privacy-preserving
release, but it severely limits semantic feature engineering — a point we return to
in the report.

In [3]:
print("Dtypes:\n", raw.dtypes.value_counts())
print("\nColumns:", list(raw.columns))
raw.describe().T[["mean", "std", "min", "50%", "max"]]

Dtypes:
 float64    30
int64       1
Name: count, dtype: int64

Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


,mean,std,min,50%,max
Time,9.481386e+04,47488.145955,0.000000,84692.000000,172792.000000
V1,1.759061e-12,1.958696,-56.407510,0.018109,2.454930
V2,-8.251130e-13,1.651309,-72.715728,0.065486,22.057729
V3,-9.654937e-13,1.516255,-48.325589,0.179846,9.382558
V4,8.321385e-13,1.415869,-5.683171,-0.019847,16.875344
V5,1.649999e-13,1.380247,-113.743307,-0.054336,34.801666
V6,4.248366e-13,1.332271,-26.160506,-0.274187,73.301626
V7,-3.054600e-13,1.237094,-43.557242,0.040103,120.589494
V8,8.777971e-14,1.194353,-73.216718,0.022358,20.007208
V9,-1.179749e-12,1.098632,-13.434066,-0.051429,15.594995


### Missing values, duplicates, and single-value features

There are **no missing values**. There are, however, **1,081 fully duplicated
rows**. Because the features are anonymised principal components, two rows being
identical across all 30 columns indicates the same transaction recorded twice.
Keeping duplicates risks the identical record appearing on both sides of the
train/test split (a subtle leak), so we drop them. No column is constant
(single-valued), so none is dropped on that basis.

In [4]:
print("Missing values total:", int(raw.isnull().sum().sum()))
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Constant (single-value) columns:", [c for c in raw.columns if raw[c].nunique() == 1])

df = data.load_data(drop_duplicates=True)
print("\nShape after dropping duplicates:", df.shape)

Missing values total: 0


Exact duplicate rows: 1081
Constant (single-value) columns: []



Shape after dropping duplicates: (283726, 31)


### Temporal coverage

`Time` runs from 0 to 172,792 seconds — almost exactly **48 hours**. The dataset is
therefore a two-day window of one bank's traffic. This matters: any temporal pattern
we find (e.g. a day/night cycle) is estimated from only two days and should be
treated cautiously.

In [5]:
print("Time min/max (seconds):", df["Time"].min(), df["Time"].max())
print("Span in hours:", round(df["Time"].max() / 3600, 1))
print("Span in days:", round(df["Time"].max() / 86400, 2))

Time min/max (seconds): 0.0 172792.0
Span in hours: 48.0
Span in days: 2.0


## 2. Exploratory Data Analysis

### 2.1 Class imbalance (prevalence)

Fraud is **0.17%** of all transactions. In real terms a naive classifier that always
predicts "legitimate" would be 99.83% accurate while catching zero fraud — which is
exactly why accuracy is the wrong headline metric here. The imbalance also means the
*positive* class is data-starved: with so few frauds, variance of any fraud-specific
estimate is high. The original tutorial addresses this with SMOTE; we will show that
*where* SMOTE is applied determines whether it helps or silently cheats.

In [6]:
balance = eda.class_balance(df)
print(balance)
print("\nFraud prevalence: %.4f%%" % (100 * balance.loc[1, "prevalence"]))
eda.plot_class_balance(df)

        count  prevalence
Class                    
0      283253    0.998333
1         473    0.001667

Fraud prevalence: 0.1667%


'/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/reports/figures/class_balance.png'

![class balance](reports/figures/class_balance.png)

### 2.2 Feature distributions and outliers

`Amount` is strongly right-skewed (median 22, max ~25,700). A `log1p` transform
tames the tail, which helps linear/distance-based models and makes the distribution
legible. Under the 1.5×IQR Tukey rule, ~11% of `Amount` values are "outliers" — but
in fraud detection extreme values are signal, not noise, so we must scale robustly
rather than clip.

In [7]:
print("Amount summary:\n", df["Amount"].describe())
print("\nIQR-outlier share of Amount: %.3f" % eda.outlier_share_iqr(df["Amount"]))
eda.plot_amount_distribution(df)

Amount summary:
 count    283726.000000
mean         88.472687
std         250.399437
min           0.000000
25%           5.600000
50%          22.000000
75%          77.510000
max       25691.160000
Name: Amount, dtype: float64

IQR-outlier share of Amount: 0.112


'/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/reports/figures/amount_distribution.png'

![amount distribution](reports/figures/amount_distribution.png)

### 2.3 Temporal analysis: fraud rate by hour of day

Converting `Time` to hour-of-day reveals that legitimate traffic follows a clear
day/night cycle, while fraud is comparatively flatter — so the *fraud rate* spikes
during the quiet overnight hours. This aligns with world knowledge: attackers
operate when monitoring is thin and victims are asleep. (Caveat: only two days of
data, so this is suggestive, not conclusive.)

In [8]:
df = features.add_hour_of_day(df)
df = features.add_log_amount(df)
print(eda.fraud_rate_by_hour(df))
eda.plot_fraud_rate_by_hour(df)

          n  fraud_rate
Hour                   
0      7647    0.000785
1      4208    0.002376
2      3308    0.014510
3      3487    0.004875
4      2204    0.010436
5      2988    0.003681
6      4082    0.002205
7      7233    0.003180
8     10232    0.000880
9     15767    0.001015
10    16548    0.000483
11    16781    0.003158
12    15378    0.001105
13    15323    0.001109
14    16520    0.001392
15    16374    0.001588
16    16396    0.001342
17    16130    0.001736
18    16959    0.001651
19    15566    0.001221
20    16705    0.001078
21    17629    0.000908
22    15378    0.000585
23    10883    0.001562


'/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/reports/figures/fraud_rate_by_hour.png'

![fraud rate by hour](reports/figures/fraud_rate_by_hour.png)

### 2.4 Correlation analysis — and why Spearman

We studied Pearson, Spearman, and Kendall correlation. The choice matters:

- **Pearson** measures *linear* association and assumes roughly normal, outlier-free
  data. `Amount` is heavily skewed and the `V` features are not Gaussian, so Pearson
  is a poor fit.
- **Spearman** measures *monotonic* association via ranks. It is robust to skew and
  outliers and makes no linearity assumption — the best general-purpose choice here.
- **Kendall** is also rank-based and preferable for very small samples or many ties;
  with 280k rows it mostly agrees with Spearman but is far more expensive to compute.

We therefore use **Spearman** as the primary measure. The correlations of individual
features with `Class` are *small in absolute terms* (top |ρ| ≈ 0.06). This is an
important, practically significant finding: **no single feature separates fraud**,
so the task is inherently multivariate and linear models will struggle relative to
tree ensembles that capture interactions. Statistical significance (large *n* makes
almost everything "significant") is not the same as practical significance.

In [9]:
spearman = eda.correlation_with_target(df.drop(columns=["Hour", "LogAmount"]), method="spearman")
print("Top features by |Spearman correlation| with Class:")
print(spearman.head(10).round(4))
eda.plot_correlation_with_target(df.drop(columns=["Hour", "LogAmount"]))

Top features by |Spearman correlation| with Class:
V14   -0.0632
V4     0.0617
V12   -0.0614
V11    0.0586
V10   -0.0580
V3    -0.0577
V2     0.0494
V16   -0.0483
V9    -0.0482
V7    -0.0466
Name: Class, dtype: float64


'/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/reports/figures/correlation_with_target.png'

![correlation with target](reports/figures/correlation_with_target.png)

### 2.5 Redundancy check

Because `V1`–`V28` are principal components, they are **mutually orthogonal by
construction**, so we expect almost no inter-feature redundancy among them. We verify
this: the largest absolute off-diagonal Spearman correlation among the `V` features
is tiny. The only redundancy we introduce ourselves is `Amount` vs `LogAmount` (a
deterministic transform) and `Time` vs `Hour`; we keep one representation per concept
in modelling to avoid duplicated signal.

In [10]:
v_corr = df[PCA_FEATURES].corr(method="spearman").abs()
np.fill_diagonal(v_corr.values, 0)
print("Max |Spearman| correlation between any two V-features: %.4f" % v_corr.values.max())

Max |Spearman| correlation between any two V-features: 0.6775


## 3. Feature Engineering

The dataset is already heavily engineered by the publisher (PCA on the original,
undisclosed features), which constrains what we can add:

- **Scaling.** `Amount` and `Time` are on very different scales from the unit-variance
  `V` components. We scale with **RobustScaler** (median/IQR) because it resists the
  `Amount` outliers that a StandardScaler would let dominate. Crucially, scaling is
  placed *inside* the modelling pipeline so it is fitted on training folds only.
- **`LogAmount`** = `log1p(Amount)` to compress the skew.
- **`Hour`** from `Time` to expose the daily cycle (Section 2.3).
- **Encoding.** There are no categorical features, so one-hot / target encoding are
  not applicable; we explain this in the report rather than forcing an encoder.
- **Dimensionality reduction.** PCA is already applied; re-applying it (as the source
  does) is redundant and, when fitted on the full data, is another leakage vector.

In [11]:
feature_columns = PCA_FEATURES + ["Amount", "Time", "Hour", "LogAmount"]
X = df[feature_columns]
y = df[TARGET]
print("Feature matrix shape:", X.shape)
print("Engineered columns added:", ["Hour", "LogAmount"])

Feature matrix shape: (283726, 32)
Engineered columns added: ['Hour', 'LogAmount']


## 4. Model Training

### 4.1 Flawed reproduction — SMOTE applied *before* the split

We first reproduce the methodology that yields the headline numbers: scale and
**SMOTE the entire dataset**, *then* split into train/test and evaluate. Because
SMOTE interpolates new minority points from existing ones across the whole dataset,
synthetic neighbours of test-set frauds end up in training, and the test set is no
longer an honest held-out sample.

In [12]:
y_true_flawed, y_pred_flawed, y_score_flawed = models.flawed_reproduction(X, y)
flawed_metrics = evaluate.classification_metrics(y_true_flawed, y_pred_flawed, y_score_flawed)
print("Flawed pipeline metrics (SMOTE before split, evaluated on resampled test set):")
pd.Series(flawed_metrics).round(4)

/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Flawed pipeline metrics (SMOTE before split, evaluated on resampled test set):


accuracy     0.9999
precision    0.9999
recall       1.0000
f1           0.9999
f2           1.0000
mcc          0.9999
roc_auc      1.0000
pr_auc       1.0000
dtype: float64

Every metric is ≈ **1.00**, faithfully reproducing the source's claim. This is *not*
evidence of a good model — it is the signature of data leakage on a synthetically
balanced test set.

### 4.2 Corrected, leakage-free pipeline

Now the honest setup:
1. **Stratified split first** (80/20), preserving the 0.17% prevalence in both parts.
2. A single `imblearn` **Pipeline** `RobustScaler → SMOTE → estimator`, evaluated with
   **StratifiedKFold(5)** cross-validation so SMOTE is fitted on each training fold
   only.
3. The 20% test set is **never resampled** and is touched once, at the end.

We compare Logistic Regression and Random Forest (matching the source) and add an
**Isolation Forest** — an unsupervised anomaly detector — as a supervised-vs-
unsupervised contrast.

In [13]:
X_train, X_test, y_train, y_test = models.stratified_split(X, y)
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Test frauds:", int(y_test.sum()))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["average_precision", "roc_auc", "recall", "precision"]

cv_rows = {}
test_results = {}
pr_curves = {}
roc_curves = {}

for name, estimator in models.supervised_estimators().items():
    pipeline = models.build_corrected_pipeline(estimator)
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows[name] = {
        "cv_PR_AUC": scores["test_average_precision"].mean(),
        "cv_ROC_AUC": scores["test_roc_auc"].mean(),
        "cv_recall": scores["test_recall"].mean(),
    }
    y_pred, y_score = models.fit_predict_proba(pipeline, X_train, y_train, X_test)
    test_results[name] = evaluate.classification_metrics(y_test, y_pred, y_score)
    pr_curves[name] = (y_test.values, y_score)
    roc_curves[name] = (y_test.values, y_score)

print("Cross-validated scores on the training set (leakage-free):")
pd.DataFrame(cv_rows).T.round(4)

Train: (226980, 32) | Test: (56746, 32) | Test frauds: 95


/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data

/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data

/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Cross-validated scores on the training set (leakage-free):


,cv_PR_AUC,cv_ROC_AUC,cv_recall
Logistic Regression,0.7543,0.9814,0.9021
Random Forest,0.8470,0.9768,0.8227


### 4.3 Isolation Forest (unsupervised)

Isolation Forest learns nothing from labels; it isolates points that are easy to
separate by random splits. We set `contamination` to the training fraud prevalence
so its default threshold is reasonable, then score the same untouched test set.

In [14]:
prevalence = float(y_train.mean())
y_pred_if, y_score_if = models.isolation_forest_scores(X_train, X_test, contamination=prevalence)
test_results["Isolation Forest"] = evaluate.classification_metrics(y_test, y_pred_if, y_score_if)
pr_curves["Isolation Forest"] = (y_test.values, y_score_if)
roc_curves["Isolation Forest"] = (y_test.values, y_score_if)
print("Isolation Forest scored on the untouched test set.")

Isolation Forest scored on the untouched test set.


## 5. Evaluation

### 5.1 Why these metrics

- **Accuracy** — fraction correct. Reported only to expose its uselessness here:
  the all-legit baseline already scores 99.83%.
- **Precision** = TP/(TP+FP) — of the transactions we flag, how many are truly fraud.
  Low precision means analysts drown in false alarms.
- **Recall** = TP/(TP+FN) — of all frauds, how many we catch. Low recall means money
  lost.
- **F1 / F2** — harmonic mean of precision and recall; **F2** weights recall higher,
  matching the fraud reality that a missed fraud (FN) costs more than a false alarm.
- **MCC** (Matthews) — a balanced correlation between predictions and truth that stays
  honest under extreme imbalance; arguably the single best scalar summary here.
- **ROC-AUC** — ranking quality across all thresholds, but optimistic under imbalance
  because the huge true-negative count flatters the false-positive rate.
- **PR-AUC / Average Precision** — area under the precision-recall curve; the most
  informative threshold-free metric for rare positives.

In [15]:
summary_table = evaluate.metrics_table(test_results)
print("Corrected-pipeline metrics on the untouched test set:")
summary_table

Corrected-pipeline metrics on the untouched test set:


,accuracy,precision,recall,f1,f2,mcc,roc_auc,pr_auc
Logistic Regression,0.9747,0.0551,0.8737,0.1038,0.2202,0.2159,0.9634,0.6859
Random Forest,0.9995,0.9125,0.7684,0.8343,0.7935,0.8371,0.9625,0.8079
Isolation Forest,0.9974,0.2022,0.1895,0.1957,0.1919,0.1945,0.9384,0.1070


### 5.2 Flawed vs corrected, side by side

The contrast is the whole point: the flawed pipeline reports ≈1.00 across the board,
while the honest Random Forest achieves a strong-but-imperfect PR-AUC ≈ 0.81 and
MCC ≈ 0.84. Note that Random Forest's *accuracy* is 0.9995 even when honest — and
Logistic Regression's accuracy is 0.97 despite catching fraud with only ~5% precision.
Accuracy simply cannot distinguish these very different models.

In [16]:
comparison = pd.DataFrame(
    {
        "Flawed RF (leaked)": pd.Series(flawed_metrics),
        "Corrected RF": pd.Series(test_results["Random Forest"]),
        "Corrected LogReg": pd.Series(test_results["Logistic Regression"]),
    }
).round(4)
comparison

,Flawed RF (leaked),Corrected RF,Corrected LogReg
accuracy,0.9999,0.9995,0.9747
precision,0.9999,0.9125,0.0551
recall,1.0000,0.7684,0.8737
f1,0.9999,0.8343,0.1038
f2,1.0000,0.7935,0.2202
mcc,0.9999,0.8371,0.2159
roc_auc,1.0000,0.9625,0.9634
pr_auc,1.0000,0.8079,0.6859


### 5.3 Curves

In [17]:
evaluate.plot_pr_curves(pr_curves)
evaluate.plot_roc_curves(roc_curves)
evaluate.plot_confusion(y_test, models.fit_predict_proba(
    models.build_corrected_pipeline(models.supervised_estimators()["Random Forest"]),
    X_train, y_train, X_test)[0], "Random Forest (corrected) confusion matrix", "confusion_rf.png")
print("Saved PR, ROC, and confusion-matrix figures to reports/figures/.")

/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Saved PR, ROC, and confusion-matrix figures to reports/figures/.


![pr curves](reports/figures/pr_curves.png)

![roc curves](reports/figures/roc_curves.png)

![confusion rf](reports/figures/confusion_rf.png)

The ROC curves look almost equally excellent for every model (all AUC > 0.93),
whereas the PR curves separate them sharply — visual proof that ROC-AUC hides the
differences that matter under imbalance.

## 6. Error Analysis

### 6.1 The false-positive / false-negative trade-off

At the default 0.5 threshold the corrected Random Forest is precision-heavy (few false
alarms, but it misses ~23% of frauds). In fraud, a missed fraud usually costs far more
than a false alarm, so we sweep the threshold with an asymmetric cost (FN = 100×FP) and
pick the cost-minimising operating point.

In [18]:
rf_pipeline = models.build_corrected_pipeline(models.supervised_estimators()["Random Forest"])
_, rf_score = models.fit_predict_proba(rf_pipeline, X_train, y_train, X_test)
cost_table = evaluate.threshold_cost_analysis(y_test, rf_score, fn_cost=100, fp_cost=1)
best = cost_table.loc[cost_table["cost"].idxmin()]
print("Cost-optimal operating point (FN=100x FP):")
print(best[["threshold", "recall", "precision", "tp", "fp", "fn"]])
evaluate.plot_cost_curve(cost_table)

/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Cost-optimal operating point (FN=100x FP):
threshold      0.090000
recall         0.852632
precision      0.336100
tp            81.000000
fp           160.000000
fn            14.000000
Name: 8, dtype: float64


'/Users/rzidane/Desktop/wael-hw/cyber/credit-card-fraud-critical-review/reports/figures/cost_curve.png'

![cost curve](reports/figures/cost_curve.png)

Lowering the threshold from 0.5 to ≈0.09 trades a handful of extra false alarms for
several additional caught frauds — the right call under the stated cost model. This is
the kind of deliberate, cost-aware decision the original tutorial never surfaces,
because its ≈1.00 metrics leave no apparent trade-off to make.

### 6.2 What kind of frauds slip through

We inspect the false negatives of the corrected Random Forest. Missed frauds skew
toward **small amounts** that look like ordinary spending — economically the least
damaging individually, but a known tactic ("card testing") for probing stolen cards
with tiny purchases before a large one.

In [19]:
rf_pred_default = (rf_score >= 0.5).astype(int)
test_view = X_test.copy()
test_view["true"] = y_test.values
test_view["pred"] = rf_pred_default
false_neg = test_view[(test_view["true"] == 1) & (test_view["pred"] == 0)]
true_pos = test_view[(test_view["true"] == 1) & (test_view["pred"] == 1)]
print("False negatives:", len(false_neg), "| True positives:", len(true_pos))
print("Median Amount — missed frauds: %.2f | caught frauds: %.2f"
      % (false_neg["Amount"].median(), true_pos["Amount"].median()))

False negatives: 22 | True positives: 73
Median Amount — missed frauds: 2.00 | caught frauds: 19.04


## 7. Conclusions

- **The source's central claim is not supported.** Its near-perfect AUC-ROC ≈ 1.00 is
  reproducible only when SMOTE is applied before the split and the model is evaluated
  on a synthetically balanced test set — i.e. it measures leakage, not skill.
- **Honest performance is strong but bounded.** A leakage-free Random Forest reaches
  PR-AUC ≈ 0.81 and MCC ≈ 0.84; it catches ~77% of frauds at 91% precision out of the
  box, and can be tuned for higher recall at a controlled cost.
- **Metric choice changes the story.** Accuracy and ROC-AUC are near-perfect for every
  model; only PR-AUC, MCC, and F2 reveal the real, large gaps between them.
- **No single feature detects fraud** (top |Spearman| ≈ 0.06); the problem is
  irreducibly multivariate, which is why tree ensembles beat the linear model.
- **Future work:** cost-sensitive learning, gradient boosting (XGBoost/LightGBM),
  threshold calibration per business cost, and a temporally-aware split if more than
  two days of data become available.

**Recommendation:** the *modelling idea* (tree ensemble on these features) is sound and
worth reusing; the *evaluation methodology* of the source is not, and its headline
numbers should not be trusted or repeated.